# PLS_Regression — teste final walk-forward

Parâmetros congelados, mesmas origens dos quatro modelos, horizonte de 1 hora e reajuste em **toda origem**. Esta é uma execução pesada.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
params = load_json(require_file(
    config.RESULT_DIR / "best_params_PLS_Regression.json",
    "01_otimizacao_hiperparametros.ipynb",
))
origins = models.final_origins(model_df)
print("Parâmetros:", params)
print("Origens finais:", len(origins))
res = models.evaluate(model_df, origins, "PLS_Regression", params)

res.feature_importance = models.pls_importance(
    model_df, params, models.validation_origins(model_df)
)

models.save_result(res, config.RESULT_DIR)

# Diagnóstico residual em 168 horas consecutivas. Essas previsões não alteram
# o MAE oficial nem a seleção; servem para ACF e Ljung-Box com espaçamento 1h.
diagnostic_origins = models.residual_diagnostic_origins(model_df)
diagnostic = models.evaluate(
    model_df, diagnostic_origins, "PLS_Regression", params, progress_every=24
)
diagnostic.predictions.to_csv(
    config.RESULT_DIR / "residual_diagnostic_PLS_Regression.csv", index=False
)

baseline_path = config.RESULT_DIR / "baseline_predictions.csv"
baseline = models.baseline_predictions(model_df, origins)
baseline.to_csv(baseline_path, index=False)

print(f"MAE final = {res.mae:.3f}")
print(f"Tempo = {res.runtime_sec / 60:.1f} min")
print(f"Diagnóstico residual consecutivo = {diagnostic.runtime_sec / 60:.1f} min")
display(res.predictions.head())
display(res.predictions.tail())


Parâmetros: {'feature_set': 'no_weather', 'n_components': 15}
Origens finais: 521
PLS_Regression: 10/521 origens | MAE parcial=167.570
PLS_Regression: 20/521 origens | MAE parcial=162.065
PLS_Regression: 30/521 origens | MAE parcial=197.082
PLS_Regression: 40/521 origens | MAE parcial=188.274
PLS_Regression: 50/521 origens | MAE parcial=195.222
PLS_Regression: 60/521 origens | MAE parcial=191.589
PLS_Regression: 70/521 origens | MAE parcial=202.867
PLS_Regression: 80/521 origens | MAE parcial=199.677
PLS_Regression: 90/521 origens | MAE parcial=219.137
PLS_Regression: 100/521 origens | MAE parcial=212.006
PLS_Regression: 110/521 origens | MAE parcial=221.805
PLS_Regression: 120/521 origens | MAE parcial=223.844
PLS_Regression: 130/521 origens | MAE parcial=219.132
PLS_Regression: 140/521 origens | MAE parcial=212.603
PLS_Regression: 150/521 origens | MAE parcial=212.024
PLS_Regression: 160/521 origens | MAE parcial=221.813
PLS_Regression: 170/521 origens | MAE parcial=219.216
PLS_Regre

,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
0,22908,2018-02-04 07:00:00,2018-02-04 08:00:00,1,1857.0,2007.873001,-150.873001,0.049144,True,False,True
1,22919,2018-02-04 18:00:00,2018-02-04 19:00:00,1,1720.0,1698.219925,21.780075,0.029443,True,False,True
2,22930,2018-02-05 05:00:00,2018-02-05 06:00:00,1,4846.0,4691.490297,154.509703,0.040774,True,False,True
3,22941,2018-02-05 16:00:00,2018-02-05 17:00:00,1,5912.0,5203.200172,708.799828,0.020498,True,False,True
4,22952,2018-02-06 03:00:00,2018-02-06 04:00:00,1,850.0,1079.566967,-229.566967,0.017007,True,False,True


,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
516,28584,2018-09-28 19:00:00,2018-09-28 20:00:00,1,3538.0,3546.117397,-8.117397,0.014496,True,False,True
517,28595,2018-09-29 06:00:00,2018-09-29 07:00:00,1,2036.0,2381.847331,-345.847331,0.014816,True,False,True
518,28606,2018-09-29 17:00:00,2018-09-29 18:00:00,1,4569.0,4592.275430,-23.275430,0.016036,True,False,True
519,28617,2018-09-30 04:00:00,2018-09-30 05:00:00,1,444.0,400.688880,43.311120,0.016965,True,False,True
520,28628,2018-09-30 15:00:00,2018-09-30 16:00:00,1,4283.0,4434.035919,-151.035919,0.020464,True,False,True


O arquivo de métricas registra o número de previsões e o tempo médio por origem. Não compare o MAE bruto com bases de outra escala.